In [1]:
# Import required libraries
import scanpy as sc

# Load the .h5ad file
adata = sc.read_h5ad("Ma2019_Liver-Biliary.h5ad")

# Print basic information about the dataset
print("Basic info about the dataset:")
print(adata)

# Show shape (cells x genes)
print("\nShape of the data (cells x genes):")
print(adata.shape)

# Show observations (cell metadata)
print("\nCell metadata (adata.obs):")
print(adata.obs.head())
print("\nColumns in adata.obs:")
print(adata.obs.columns)

# Show variables (gene metadata)
print("\nGene metadata (adata.var):")
print(adata.var.head())
print("\nColumns in adata.var:")
print(adata.var.columns)

# Show available layers
print("\nAvailable layers:")
print(adata.layers.keys())

# Show embeddings (like PCA, UMAP)
print("\nAvailable embeddings (obsm):")
print(adata.obsm.keys())

# Show unstructured annotations
print("\nUnstructured data (uns):")
print(adata.uns.keys())

# Check raw data
if adata.raw is not None:
    print("\nRaw data is available")
    print(adata.raw)
else:
    print("\nNo raw data found")

# Show first few expression values
print("\nFirst few values of expression matrix:")
print(adata.X[:5, :5])

Basic info about the dataset:
AnnData object with n_obs × n_vars = 2463 × 17505
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'cell_type', 'complexity', 'umap1', 'umap2', 'g1s_score', 'g2m_score', 'cell_cycle_phase', 'mp_top_score', 'mp_top', 'mp_assignment', 'disease', 'nCount_tmp', 'nFeature_tmp', 'percent.ribo', 'percent.mito', 'log10GenesPerUmi', 'ident'
    uns: 'X_name'
    obsm: 'PCA', 'UMAP'
    layers: 'logcounts'

Shape of the data (cells x genes):
(2463, 17505)

Cell metadata (adata.obs):
                                                               orig.ident  \
AAACCTGAGATAGCAT-3,C26,Endothelial,1247,-16.72,...  Ma2019_Liver-Biliary_   
AAACCTGAGATGTAAC-9,H38,Malignant,3524,29.304,15...  Ma2019_Liver-Biliary_   
AAACCTGAGGAATTAC-9,H38,Fibroblast,1562,10.737,-...  Ma2019_Liver-Biliary_   
AAACCTGAGGTACTCT-22,C66,Malignant,2446,14.063,-...  Ma2019_Liver-Biliary_   
AAACCTGCAATGGACG-13,H37,Malignant,4797,6.3469,4...  Ma2019_Liver-Biliary_   

               

In [16]:
import scanpy as sc
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from itertools import product

# load dataset
adata = sc.read_h5ad("Ma2019_Liver-Biliary.h5ad")

samples = adata.obs["sample"].unique()

# split C and H samples
C_samples = [s for s in samples if s.startswith("C")]
H_samples = [s for s in samples if s.startswith("H")]

results = []

for c, h in product(C_samples, H_samples):

    train_samples = [c, h]
    test_samples = [s for s in samples if s not in train_samples]

    train_mask = adata.obs["sample"].isin(train_samples)
    test_mask = adata.obs["sample"].isin(test_samples)

    X_train = adata[train_mask].X
    X_test = adata[test_mask].X

    y_train = adata[train_mask].obs["cell_type"]
    y_test = adata[test_mask].obs["cell_type"]

    # convert sparse to dense
    if hasattr(X_train, "toarray"):
        X_train = X_train.toarray()
        X_test = X_test.toarray()

    rf = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

    results.append({
        "train_C": c,
        "train_H": h,
        "accuracy": acc
    })

    print(f"Train: ({c}, {h}) -> Accuracy: {acc:.4f}")

# final aggregation
import pandas as pd
df = pd.DataFrame(results)

print("\n=== FINAL RESULTS ===")
print(df)

print("\nMean Accuracy:", df["accuracy"].mean())
print("Std Accuracy:", df["accuracy"].std())

Train: (C26, H38) -> Accuracy: 0.8675
Train: (C26, H37) -> Accuracy: 0.7421
Train: (C26, H65) -> Accuracy: 0.6540
Train: (C66, H38) -> Accuracy: 0.9805
Train: (C66, H37) -> Accuracy: 0.8315
Train: (C66, H65) -> Accuracy: 0.8604
Train: (C46, H38) -> Accuracy: 0.9799
Train: (C46, H37) -> Accuracy: 0.8024
Train: (C46, H65) -> Accuracy: 0.8044
Train: (C56, H38) -> Accuracy: 0.8572
Train: (C56, H37) -> Accuracy: 0.6165
Train: (C56, H65) -> Accuracy: 0.6959
Train: (C25, H38) -> Accuracy: 0.9866
Train: (C25, H37) -> Accuracy: 0.6305
Train: (C25, H65) -> Accuracy: 0.6891

=== FINAL RESULTS ===
   train_C train_H  accuracy
0      C26     H38  0.867550
1      C26     H37  0.742141
2      C26     H65  0.654036
3      C66     H38  0.980539
4      C66     H37  0.831492
5      C66     H65  0.860400
6      C46     H38  0.979945
7      C46     H37  0.802380
8      C46     H65  0.804362
9      C56     H38  0.857239
10     C56     H37  0.616545
11     C56     H65  0.695942
12     C25     H38  0.986640
1

In [2]:
# =========================
# Imports
# =========================
import scanpy as sc
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import classification_report, roc_curve, auc, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import matplotlib.pyplot as plt
import seaborn as sns

# =========================
# Create structured folders
# =========================
base_dir = "results"
subdirs = [
    "classification_report",
    "roc_curves",
    "auc_tables",
    "cross_validation",
    "confusion_matrix",
    "feature_importance"
]

for sub in subdirs:
    os.makedirs(os.path.join(base_dir, sub), exist_ok=True)

# =========================
# Load data
# =========================
adata = sc.read_h5ad("Ma2019_Liver-Biliary.h5ad")

# Use logcounts if available
if "logcounts" in adata.layers:
    adata.X = adata.layers["logcounts"]

# =========================
# HVG Feature Selection
# =========================
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
adata = adata[:, adata.var.highly_variable]

gene_names = adata.var_names

# =========================
# Prepare data
# =========================
X = adata.X
X = X.toarray() if not isinstance(X, np.ndarray) else X

y = adata.obs["cell_type"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)
class_names = le.classes_

# =========================
# Stratified Split (no leakage)
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

# =========================
# Models
# =========================
models = {
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "XGBoost": XGBClassifier(eval_metric="mlogloss", use_label_encoder=False),
    "CatBoost": CatBoostClassifier(verbose=0)
}

# =========================
# ROC prep
# =========================
y_test_bin = label_binarize(y_test, classes=np.unique(y_encoded))
n_classes = y_test_bin.shape[1]

# =========================
# Main loop
# =========================
all_reports = []

for name, model in models.items():
    print(f"Training {name}...")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # -------------------------
    # Classification Report
    # -------------------------
    report = classification_report(
        y_test, y_pred, target_names=class_names, output_dict=True
    )
    df_report = pd.DataFrame(report).transpose()
    df_report.to_csv(f"{base_dir}/classification_report/{name}.csv")

    # -------------------------
    # ROC + AUC
    # -------------------------
    y_score = model.predict_proba(X_test)

    plt.figure()
    auc_list = []

    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)

        plt.plot(fpr, tpr, label=f"{class_names[i]} (AUC={roc_auc:.2f})")

        auc_list.append({
            "model": name,
            "cell_type": class_names[i],
            "auc": roc_auc
        })

    plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.title(f"ROC - {name}")
    plt.legend()
    plt.savefig(f"{base_dir}/roc_curves/{name}.png")
    plt.close()

    pd.DataFrame(auc_list).to_csv(f"{base_dir}/auc_tables/{name}.csv", index=False)

    # -------------------------
    # Confusion Matrix
    # -------------------------
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=class_names,
                yticklabels=class_names)

    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion Matrix - {name}")
    plt.xticks(rotation=45)
    plt.yticks(rotation=45)

    plt.savefig(f"{base_dir}/confusion_matrix/{name}.png")
    plt.close()

    # -------------------------
    # Feature Importance
    # -------------------------
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_

        fi_df = pd.DataFrame({
            "gene": gene_names,
            "importance": importances
        }).sort_values(by="importance", ascending=False)

        fi_df.head(50).to_csv(f"{base_dir}/feature_importance/{name}_top50.csv", index=False)

# =========================
# Cross Validation (train only)
# =========================
cv_results = []

skf5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skf10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

for name, model in models.items():
    print(f"CV for {name}...")

    cv5 = cross_val_score(model, X_train, y_train, cv=skf5)
    cv10 = cross_val_score(model, X_train, y_train, cv=skf10)

    cv_results.append({
        "model": name,
        "cv5_mean": np.mean(cv5),
        "cv5_std": np.std(cv5),
        "cv10_mean": np.mean(cv10),
        "cv10_std": np.std(cv10)
    })

pd.DataFrame(cv_results).to_csv(f"{base_dir}/cross_validation/cv_results.csv", index=False)

print("✅ DONE! Everything saved in structured folders.")

Training RandomForest...


C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average,

Training DecisionTree...
Training XGBoost...


C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:26:32] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Arman\AppData\

Training CatBoost...


...

CV for XGBoost...


C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:29:48] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:29:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:29:53] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters

CV for CatBoost...


C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
C:\Users\Arman\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=10.
  warnings.warn(


✅ DONE! Everything saved in structured folders.
